In [1]:
%reload_ext autoreload
%autoreload 1

# M5 Forecast

## Imports

In [4]:
import time
import numpy as np
import multiprocessing
import pandas as pd
from pathlib import Path

id_col = "id"
time_col = "date"
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']

PATH_INPUT = Path("data/m5/datasets")
N_JOBS = multiprocessing.cpu_count()

HORIZON = 28  # How many days into the future to predict
VAL_DAYS = 0  # How many days of data to hold-out for validatioin (set to 0 for live)

if not PATH_INPUT.exists():
    raise FileNotFoundError("Please download the M5 dataset first by executing `src/generate_data.py`")
    
os.environ["NIXTLA_ID_AS_COL"] = '1'
start_time = time.time()

# Helper Functions

In [5]:
start_time = time.time()

from typing import List
import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.lag_transforms import ExpandingMean, RollingMean, SeasonalRollingMean
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mse, mase, mape, mae, smape
from utilsforecast.plotting import plot_series
from utilsforecast.preprocessing import fill_gaps
from src.process import *

def is_first_or_fifteenth(dates):
    """Date is the 1st or 15th of the month"""
    return dates.day.isin([1, 15])

def even_day(dates):
    """Day of month is even"""
    return dates.day % 2 == 0

def month_start_or_end(dates):
    """Date is month start or month end"""
    return dates.is_month_start | dates.is_month_end

def is_monday(dates):
    """Date is monday"""
    return dates.dayofweek == 0

def downcast_id_col(df: pd.DataFrame, 
                    df_ids: pd.DataFrame, 
                    id_col: str = id_col):
    """Downcast df[id_col] to integer.
    """
    df[id_col] = df[id_col].map(df_ids.set_index("id")["idx"].to_dict())

def is_business_day(dates):
    """Date is a business day (Monday to Friday)"""
    return dates.dt.dayofweek < 5  # Monday=0, Sunday=6

def is_quarter_start(dates):
    """Date is quarter start"""
    return dates.dt.is_quarter_start

def even_day(dates):
    """Day of month is even"""
    return dates.dt.day % 2 == 0

def month_start_or_end(dates):
    """Date is month start or month end"""
    return dates.dt.is_month_start | dates.dt.is_month_end

def is_monday(dates):
    """Date is Monday"""
    return dates.dt.dayofweek == 0

def is_weekend(dates):
    """Date is on a weekend (Saturday or Sunday)"""
    return dates.dt.dayofweek >= 5

def is_week_start(dates):
    """Date is the start of the week (Monday)"""
    return dates.dt.dayofweek == 0

def is_week_end(dates):
    """Date is the end of the week (Sunday)"""
    return dates.dt.dayofweek == 6

def is_year_start_or_end(dates):
    """Date is year start or year end"""
    return dates.dt.is_year_start | dates.dt.is_year_end

def is_leap_year(dates):
    """Date is in a leap year"""
    return dates.dt.is_leap_year

def days_until_month_end(dates):
    """Number of days until the end of the month"""
    month_end = dates + pd.offsets.MonthEnd(0)
    return (month_end - dates).dt.days

def is_fiscal_year_end(dates, month=12):
    """Date is the end of the fiscal year
    Args:
        dates (pd.Series): Series of datetime objects.
        month (int): Month considered as the fiscal year-end.
    """
    return (dates.dt.month == month) & dates.dt.is_month_end

def day_of_quarter(dates):
    """Day number within the quarter"""
    quarter_start = dates.dt.to_period('Q').start_time
    return (dates - quarter_start).dt.days + 1

def week_of_month(dates):
    """Week number within the month"""
    return ((dates.dt.day - 1) // 7) + 1

def is_special_date(dates, special_dates):
    """Date matches any in a list of special dates
    Args:
        dates (pd.Series): Series of datetime objects.
        special_dates (list or set): Special dates to check against.
    """
    special_dates = pd.to_datetime(special_dates)
    return dates.isin(special_dates)


def evaluate_cross_validation(df: pd.DataFrame,
                              metric: callable, 
                              id_col: str = 'unique_id',
                              time_col: str = "ds",
                             ) -> pd.DataFrame:
    """Evaluate cross-validation results.
    """
    models = df.drop(columns=[id_col, 'ds', 'cutoff', 'y']).columns.tolist()
    evals = []
    # Calculate loss for every unique_id and cutoff.    
    for cutoff in df['cutoff'].unique():
        eval_ = evaluate(df[df['cutoff'] == cutoff],
                         metrics=[metric],
                         models=models,
                         id_col=id_col,
                         time_col=time_col)
        evals.append(eval_)
    evals = pd.concat(evals)
    evals = evals.groupby(id_col).mean(numeric_only=True) # Averages the error metrics for all cutoffs for every combination of model and unique_id
    evals['best_model'] = evals.idxmin(axis=1)
    return evals


def get_best_model_forecast(forecasts_df: pd.DataFrame,
                            evaluation_df: pd.DataFrame, 
                            id_col: str = "unique_id") -> pd.DataFrame:
    """Create Production Forecast DataFrame.

    returns: df Dataframe with columns:
        id_col, ds, best_model, forecast.
    """
    df = forecasts_df.set_index([id_col, "ds"]).stack().to_frame().reset_index(level=2)
    df.columns = ['model', 'best_model_forecast']
    df = df.join(evaluation_df[['best_model']])

    df = df[df["best_model"] == df["model"]].reset_index().drop("model", axis=1)
    return df

# Load Inputs

In [6]:
df = pd.read_parquet("data/train.snap.parquet")

# Model

In [7]:
# Future Covariates
X_cols = ['sell_price', 'id', 'date', 'event_name_1', 'event_type_1',
          'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']

In [8]:
df.tail()

/home/azureuser/micromamba/envs/m5rick/lib/python3.10/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,id,item_id,dept_id,cat_id,store_id,state_id,d,y,date,wm_yr_wk,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
46796215,HOUSEHOLD_2_516_WI_3_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,d_1937,0.0,2016-05-18,11616,nan,nan,nan,nan,0,0,0,5.941406
46796216,HOUSEHOLD_2_516_WI_3_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,d_1938,0.0,2016-05-19,11616,nan,nan,nan,nan,0,0,0,5.941406
46796217,HOUSEHOLD_2_516_WI_3_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,d_1939,0.0,2016-05-20,11616,nan,nan,nan,nan,0,0,0,5.941406
46796218,HOUSEHOLD_2_516_WI_3_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,d_1940,0.0,2016-05-21,11617,nan,nan,nan,nan,0,0,0,5.941406
46796219,HOUSEHOLD_2_516_WI_3_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,d_1941,0.0,2016-05-22,11617,nan,nan,nan,nan,0,0,0,5.941406


In [9]:
df.date.max()

Timestamp('2016-05-22 00:00:00')

# Statistical Forecast
This is just a quick exploration, implement `Naive` forecast to set a benchmark for performance. There are too many time-series for the statistical models to use as a starting point.

- [statsforecast - nixtla](https://)
- [naive-methods](https://otexts.com/fpp3/simple-methods.html)
- [complex-exponential smoothing](https://onlinelibrary.wiley.com/doi/full/10.1002/nav.22074)

In [10]:
from statsforecast.models import (
    Naive, 
    WindowAverage,
    SeasonalNaive,

    # Statistical Models
    ETS,
    AutoARIMA, AutoETS, AutoCES,
    
    # Itermittent Demand Models
    CrostonOptimized,
    CrostonSBA,
    CrostonClassic as Croston,
    # IMAPA,
)
from statsforecast.models import IMAPA
from statsforecast import StatsForecast


K = 50   # How to choose seasonality for daily data

model_list = [Naive(),
              WindowAverage(window_size=7 * 1, alias="1WeekMA"),
              WindowAverage(window_size=7 * 2, alias="2WeekMA"),
              WindowAverage(window_size=7 * 4, alias="4WeekMA"),

              # Itermittment Demand Optimized Models
              Croston(),
              CrostonOptimized(),
              CrostonSBA(),

              # # IMAPA(),
              # These take far too long
              # AutoCES(season_length=K),
              # AutoETS(season_length=K),
              ETS(season_length=7, model='ZNA'),
             ]

sf = StatsForecast(models=model_list, freq="D", n_jobs=N_JOBS)

/home/azureuser/micromamba/envs/m5rick/lib/python3.10/site-packages/statsforecast/models.py:887: FutureWarning: `ETS` will be deprecated in future versions of `StatsForecast`. Please use `AutoETS` instead.
  ETS._warn()


# Cross-Validation

In [9]:
%%time

CV_WINDOWS = 3

stat_cv = sf.cross_validation(h=HORIZON,
                              n_windows=CV_WINDOWS,
                              df=df,
                              id_col=id_col, time_col=time_col
                             )

stat_cv.tail()

CPU times: user 9.08 s, sys: 7.46 s, total: 16.5 s
Wall time: 46.8 s


,id,date,cutoff,y,Naive,1WeekMA,2WeekMA,4WeekMA,CrostonClassic,CrostonOptimized,CrostonSBA
2561155,HOUSEHOLD_2_516_WI_3_evaluation,2016-05-18,2016-04-24,0.0,0.0,0.0,0.0,0.0,0.106247,0.106247,0.100935
2561156,HOUSEHOLD_2_516_WI_3_evaluation,2016-05-19,2016-04-24,0.0,0.0,0.0,0.0,0.0,0.106247,0.106247,0.100935
2561157,HOUSEHOLD_2_516_WI_3_evaluation,2016-05-20,2016-04-24,0.0,0.0,0.0,0.0,0.0,0.106247,0.106247,0.100935
2561158,HOUSEHOLD_2_516_WI_3_evaluation,2016-05-21,2016-04-24,0.0,0.0,0.0,0.0,0.0,0.106247,0.106247,0.100935
2561159,HOUSEHOLD_2_516_WI_3_evaluation,2016-05-22,2016-04-24,0.0,0.0,0.0,0.0,0.0,0.106247,0.106247,0.100935


In [138]:
stat_preds

,id,date,Naive,1WeekMA,2WeekMA,4WeekMA,CrostonClassic,CrostonOptimized,CrostonSBA
0,FOODS_1_001_CA_1_evaluation,2016-05-23,0.0,0.142857,0.642857,0.821429,0.898247,0.898247,0.853334
1,FOODS_1_001_CA_1_evaluation,2016-05-24,0.0,0.142857,0.642857,0.821429,0.898247,0.898247,0.853334
2,FOODS_1_001_CA_1_evaluation,2016-05-25,0.0,0.142857,0.642857,0.821429,0.898247,0.898247,0.853334
3,FOODS_1_001_CA_1_evaluation,2016-05-26,0.0,0.142857,0.642857,0.821429,0.898247,0.898247,0.853334
4,FOODS_1_001_CA_1_evaluation,2016-05-27,0.0,0.142857,0.642857,0.821429,0.898247,0.898247,0.853334
...,...,...,...,...,...,...,...,...,...
853715,HOUSEHOLD_2_516_WI_3_evaluation,2016-06-15,0.0,0.000000,0.071429,0.071429,0.076422,0.076422,0.072601
853716,HOUSEHOLD_2_516_WI_3_evaluation,2016-06-16,0.0,0.000000,0.071429,0.071429,0.076422,0.076422,0.072601
853717,HOUSEHOLD_2_516_WI_3_evaluation,2016-06-17,0.0,0.000000,0.071429,0.071429,0.076422,0.076422,0.072601
853718,HOUSEHOLD_2_516_WI_3_evaluation,2016-06-18,0.0,0.000000,0.071429,0.071429,0.076422,0.076422,0.072601


In [139]:
stat_preds.to_parquet("data/stat_baseline.snap.parquet")

In [167]:
stat_preds

,id,date,Naive,1WeekMA,2WeekMA,4WeekMA,CrostonClassic,CrostonOptimized,CrostonSBA
0,FOODS_1_001_CA_1_evaluation,2016-05-23,0.0,0.142857,0.642857,0.821429,0.898247,0.898247,0.853334
1,FOODS_1_001_CA_1_evaluation,2016-05-24,0.0,0.142857,0.642857,0.821429,0.898247,0.898247,0.853334
2,FOODS_1_001_CA_1_evaluation,2016-05-25,0.0,0.142857,0.642857,0.821429,0.898247,0.898247,0.853334
3,FOODS_1_001_CA_1_evaluation,2016-05-26,0.0,0.142857,0.642857,0.821429,0.898247,0.898247,0.853334
4,FOODS_1_001_CA_1_evaluation,2016-05-27,0.0,0.142857,0.642857,0.821429,0.898247,0.898247,0.853334
...,...,...,...,...,...,...,...,...,...
853715,HOUSEHOLD_2_516_WI_3_evaluation,2016-06-15,0.0,0.000000,0.071429,0.071429,0.076422,0.076422,0.072601
853716,HOUSEHOLD_2_516_WI_3_evaluation,2016-06-16,0.0,0.000000,0.071429,0.071429,0.076422,0.076422,0.072601
853717,HOUSEHOLD_2_516_WI_3_evaluation,2016-06-17,0.0,0.000000,0.071429,0.071429,0.076422,0.076422,0.072601
853718,HOUSEHOLD_2_516_WI_3_evaluation,2016-06-18,0.0,0.000000,0.071429,0.071429,0.076422,0.076422,0.072601


In [15]:
%%time

sf.fit(df[df["item_id"].isin(df.item_id.sample(100))], id_col=id_col, time_col=time_col)
sf.fit(df, id_col=id_col, time_col=time_col)
       # df[["id", "date", "y"]].rename(columns={"id": "unique_id", "date": "ds"}))
stat_preds = sf.predict(HORIZON)
stat_preds

CPU times: user 43.1 s, sys: 39.4 s, total: 1min 22s
Wall time: 5min 42s


,id,date,Naive,1WeekMA,2WeekMA,4WeekMA,CrostonClassic,CrostonOptimized,CrostonSBA,ETS
0,FOODS_1_001_CA_1_evaluation,2016-05-23,0.0,0.142857,0.642857,0.821429,0.898247,0.898247,0.853334,0.717074
1,FOODS_1_001_CA_1_evaluation,2016-05-24,0.0,0.142857,0.642857,0.821429,0.898247,0.898247,0.853334,0.685195
2,FOODS_1_001_CA_1_evaluation,2016-05-25,0.0,0.142857,0.642857,0.821429,0.898247,0.898247,0.853334,0.742611
3,FOODS_1_001_CA_1_evaluation,2016-05-26,0.0,0.142857,0.642857,0.821429,0.898247,0.898247,0.853334,0.671224
4,FOODS_1_001_CA_1_evaluation,2016-05-27,0.0,0.142857,0.642857,0.821429,0.898247,0.898247,0.853334,0.956713
...,...,...,...,...,...,...,...,...,...,...
853715,HOUSEHOLD_2_516_WI_3_evaluation,2016-06-15,0.0,0.000000,0.071429,0.071429,0.076422,0.076422,0.072601,0.021631
853716,HOUSEHOLD_2_516_WI_3_evaluation,2016-06-16,0.0,0.000000,0.071429,0.071429,0.076422,0.076422,0.072601,0.068773
853717,HOUSEHOLD_2_516_WI_3_evaluation,2016-06-17,0.0,0.000000,0.071429,0.071429,0.076422,0.076422,0.072601,0.079541
853718,HOUSEHOLD_2_516_WI_3_evaluation,2016-06-18,0.0,0.000000,0.071429,0.071429,0.076422,0.076422,0.072601,0.098033


In [16]:
stat_model_list = [str(m) for m in sf.models]
stat_model_list

['Naive',
 '1WeekMA',
 '2WeekMA',
 '4WeekMA',
 'CrostonClassic',
 'CrostonOptimized',
 'CrostonSBA',
 'ETS']

Exception ignored in atexit callback: <bound method InteractiveShell.atexit_operations of <ipykernel.zmqshell.ZMQInteractiveShell object at 0x7ff94718ea40>>
Traceback (most recent call last):
  File "/home/azureuser/micromamba/envs/m5rick/lib/python3.10/site-packages/IPython/core/interactiveshell.py", line 3908, in atexit_operations
    self.history_manager.end_session()
  File "/home/azureuser/micromamba/envs/m5rick/lib/python3.10/site-packages/IPython/core/history.py", line 576, in end_session
    self.db.execute("""UPDATE sessions SET end=?, num_cmds=? WHERE
sqlite3.OperationalError: database or disk is full


In [107]:
TRAIN_START = 1
TRAIN_END = 1941

TEST_START = 1942
TEST_END = 1969
TEST_DAYS = [f"d_{x}" for x in range(TEST_START, TEST_END + 1)]

from datasetsforecast.m5 import M5Evaluation
truth = pd.read_csv(PATH_INPUT / "sales_test_evaluation.csv")
truth["id"] = truth["item_id"].astype(str) + "_" + truth["store_id"].astype(str) + "_evaluation"

def make_submission(preds: pd.DataFrame,
                    h: int) -> pd.DataFrame:
    wide = preds.pivot_table(index='id',
                             columns='date',
                             observed=True)
    wide.columns = [f'F{i+1}' for i in range(h)]
    wide.columns.name = None
    wide.index.name = 'id'
    return wide


In [135]:
stat_models = sf.models

for model in stat_model_list:
    preds_long = stat_preds[["id", "date", str(model)]]
    df_sub = make_submission(preds_long, HORIZON)
    df_sub = truth[id_cols].merge(df_sub, on=["id"])
    model_score = M5Evaluation.evaluate("data", df_sub)
    model_score_avg = model_score["wrmsse"].mean()
    logger.info(f"""{model} scored: {model_score_avg:.3f}""")


2025-02-03 01:18:00.665 | INFO     | __main__:<cell line: 3>:9 - Naive scored: 1.752
2025-02-03 01:18:19.268 | INFO     | __main__:<cell line: 3>:9 - 1WeekMA scored: 0.955
2025-02-03 01:18:37.745 | INFO     | __main__:<cell line: 3>:9 - 2WeekMA scored: 0.963
2025-02-03 01:18:55.443 | INFO     | __main__:<cell line: 3>:9 - 4WeekMA scored: 0.946
2025-02-03 01:19:14.279 | INFO     | __main__:<cell line: 3>:9 - CrostonClassic scored: 0.957
2025-02-03 01:19:32.710 | INFO     | __main__:<cell line: 3>:9 - CrostonOptimized scored: 0.958
2025-02-03 01:19:51.147 | INFO     | __main__:<cell line: 3>:9 - CrostonSBA scored: 0.957


In [136]:
%%time

from tqdm import tqdm

stat_models = sf.models

stat_results = []
for model in tqdm(sf.models):
    model = str(model)
    preds_long = stat_preds[["id", "date", model]].copy()
    preds_long[model] = preds_long[model].round(0).clip(0)
    df_sub = make_submission(preds_long, HORIZON)
    df_sub = truth[id_cols].merge(df_sub, on=["id"])
    model_score = M5Evaluation.evaluate("data", df_sub)
    model_score["model"] = model
    model_score_avg = model_score["wrmsse"].mean()
    stat_results.append(model_score)
    logger.info(f"""{model} scored: {model_score_avg:.3f}""")

2025-02-03 01:20:09.548 | INFO     | __main__:<module>:16 - Naive scored: 1.752
2025-02-03 01:20:27.385 | INFO     | __main__:<module>:16 - 1WeekMA scored: 0.987
2025-02-03 01:20:45.958 | INFO     | __main__:<module>:16 - 2WeekMA scored: 1.004
2025-02-03 01:21:04.862 | INFO     | __main__:<module>:16 - 4WeekMA scored: 1.016
2025-02-03 01:21:22.923 | INFO     | __main__:<module>:16 - CrostonClassic scored: 1.001
2025-02-03 01:21:40.716 | INFO     | __main__:<module>:16 - CrostonOptimized scored: 1.000
2025-02-03 01:21:59.456 | INFO     | __main__:<module>:16 - CrostonSBA scored: 1.063
100%|██████████| 7/7 [02:08<00:00, 18.33s/it]

CPU times: user 1min 44s, sys: 22.7 s, total: 2min 7s
Wall time: 2min 8s


In [140]:
dfresults_stats = pd.concat(stat_results)
dfresults_stats

,wrmsse,model
Total,1.752010,Naive
Level1,1.966796,Naive
Level2,1.904404,Naive
Level3,1.879622,Naive
Level4,1.946932,Naive
...,...,...
Level8,1.057402,CrostonSBA
Level9,1.109897,CrostonSBA
Level10,1.116344,CrostonSBA
Level11,1.022634,CrostonSBA


In [142]:
dfresults_stats.groupby("model")["wrmsse"].mean().sort_values()

model
1WeekMA             0.986843
CrostonOptimized    0.999602
CrostonClassic      1.001143
2WeekMA             1.004404
4WeekMA             1.015934
CrostonSBA          1.063477
Naive               1.752010
Name: wrmsse, dtype: float64

# LightGbm CV

Fit the above MLForecast with lgb.LGBMRegressor parameters `lgb_params` specified.

- FREQ: `D` daily
- LAG_FEATURES: Calculates a list of `n` period with (freq) lag (i.e. 1, 7 would be y_1 and y_7 equal to yesterday and today. 
- LAG_TRANSFORM_DICT: Specify dictionary of tranformations to apply against `LAG_FEATURES`

- `lgb_params`: parameter dictionary for lightgbm
    - `num_leaves` and `n_estimators` are the key ones handling regularization and complexity

In [119]:
lgb_params = {
    'verbose': 1,
    'num_threads': 4,
    'force_col_wise': True,
    'num_leaves': 256,
    'n_estimators': 100,
}

N_WEEKLY_LAGS = 8
WEEKLY_LAG_FEATURES = [7 * (i+1) for i in range(8)]
LAG_FEATURES = [1, 2, 3, ] + WEEKLY_LAG_FEATURES

LAG_TRANSFORM_DICT = {
    1:  [ExpandingMean()],
    7:  [RollingMean(7), SeasonalRollingMean(7, 1)],    # Capture weekly trend ajusting for seasonality (day of week effects)
    14: [RollingMean(14), SeasonalRollingMean(14, 1)],  # Capture bi-weekly trend adjusting for seasonality (i.e. payday)
    28: [RollingMean(28), SeasonalRollingMean(28, 1)],  # Capture Monthly trend adjusting for monthly seasonality
}

# List of built-in date features to compute with MLForecast.date_features
DATE_FEATURE_LIST = [
        'year', 'month', 'day', 'day_of_week', 'quarter', 'week',
        'is_month_start', 'is_month_end', 'is_quarter_start', 'is_quarter_end',
        is_first_or_fifteenth,
    ]

In [120]:
%%time

fcst = MLForecast(
    models=[lgb.LGBMRegressor(**lgb_params)],
    freq='D',
    lags=LAG_FEATURES,
    lag_transforms = LAG_TRANSFORM_DICT,
    date_features = DATE_FEATURE_LIST,
    num_threads=4,
)

PARAMS = fcst.models["LGBMRegressor"].get_params()

lgb_cv = fcst.cross_validation(h=HORIZON,
                               n_windows=1,
                               df=df,
                               static_features=id_cols,
                               id_col=id_col, time_col=time_col
                              )

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Total Bins 34763
[LightGBM] [Info] Number of data points in the train set: 44235060, number of used features: 45
[LightGBM] [Info] Start training from score 1.425438
CPU times: user 19min 49s, sys: 30.6 s, total: 20min 19s
Wall time: 6min 21s


## Combine CV Results

- `stat_cv` - DataFrame of stats models cross-validated predictions
- `lgb_cv` - DataFrame of lightgbm cross-validated predictions
- `df_cv` - DataFrame of combined cv results all forecasts

In [121]:
df_cv = stat_cv.merge(lgb_cv[[id_col, time_col, "cutoff", "LGBMRegressor"]],
                      on=[id_col, time_col, "cutoff"])
df_cv = df_cv.rename(columns={time_col: "ds"})

MODEL_NAMES = df_cv.drop(["id", "ds", "cutoff", "y"], axis="columns").columns.tolist()

df_cv[MODEL_NAMES] = df_cv[MODEL_NAMES].round(0)  # round forecast to nearest integer

cv_err = evaluate_cross_validation(df_cv, id_col=id_col,
                                   # metric=smape
                                   metric=mae
                                  )
cv_err["LGBMRegressor"] = cv_err["LGBMRegressor"].round(0)
bst = get_best_model_forecast(df_cv, cv_err, id_col)
assert int(bst.groupby("id", observed=True).agg({"best_model": "nunique"}).max().iloc[0]) == 1
bst.best_model.value_counts(normalize=True)

/tmp/ipykernel_114725/3772746967.py:126: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  evals = evals.groupby(id_col).mean(numeric_only=True) # Averages the error metrics for all cutoffs for every combination of model and unique_id


best_model
LGBMRegressor       0.377763
Naive               0.361758
1WeekMA             0.169006
2WeekMA             0.050476
4WeekMA             0.026960
CrostonClassic      0.009905
CrostonSBA          0.003116
CrostonOptimized    0.001017
Name: proportion, dtype: float64

# Fit LightGbm on Full Data

In [122]:
%%time

lgb_start_time = time.time()

last_date_train = df['date'].max()
last_wmyrwk = df[df['date'] < last_date_train]['wm_yr_wk'].max()
logger.info(f"""Begin fitting LightGbm with {df[id_col].nunique():,d} ids data before: {last_date_train.date()}""")

logger.info(f"""LGB Fit Parameters:\n{PARAMS}""")
fcst.fit(
    df,
    id_col=id_col,
    time_col=time_col,
    target_col='y',
    static_features=id_cols,
)

logger.info(f"""Finished fitting LightGbm on {df["id"].nunique():,d} time-series in {time.time()- lgb_start_time:,.2f} seconds.""")

2025-02-02 23:55:40.620 | INFO     | __main__:<module>:5 - Begin fitting LightGbm with 30,490 ids data before: 2016-05-22
2025-02-02 23:55:40.622 | INFO     | __main__:<module>:7 - LGB Fit Parameters:
{'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 1.0, 'importance_type': 'split', 'learning_rate': 0.1, 'max_depth': -1, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 100, 'n_jobs': None, 'num_leaves': 256, 'objective': None, 'random_state': None, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 1.0, 'subsample_for_bin': 200000, 'subsample_freq': 0, 'verbose': 1, 'num_threads': 4, 'force_col_wise': True}


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Total Bins 34831
[LightGBM] [Info] Number of data points in the train set: 45088780, number of used features: 45
[LightGBM] [Info] Start training from score 1.425767


2025-02-03 00:01:43.454 | INFO     | __main__:<module>:16 - Finished fitting LightGbm on 30,490 time-series in 367.57 seconds.


CPU times: user 20min 18s, sys: 18.2 s, total: 20min 36s
Wall time: 6min 7s


In [123]:
## Create Future Regressors

cal = load_calendar(PATH_INPUT)
prices = load_prices(PATH_INPUT)

X_df = create_future_features(last_date_train, cal, prices, HORIZON)
X_df = X_df.merge(cal[["date", "d", "wm_yr_wk"]], on='date', how='left')

2025-02-03 00:01:43.464 | DEBUG    | src.process:load_calendar:30 - Begin Loading Calendar Data...
2025-02-03 00:01:43.529 | DEBUG    | src.process:load_prices:44 - Begin Loading Price Data...


In [124]:
%%time
preds = fcst.predict(HORIZON, X_df=X_df)
logger.info(f"""Finished generating {HORIZON:,d} days of predictions for {preds["id"].nunique():,d} time-series in {time.time()- lgb_start_time:,.2f} seconds.""")

2025-02-03 00:01:56.288 | INFO     | __main__:<module>:2 - Finished generating 28 days of predictions for 30,490 time-series in 380.40 seconds.


CPU times: user 18.1 s, sys: 1.75 s, total: 19.9 s
Wall time: 8.85 s


In [158]:
TEST_START 

1942

# Create Submission

In [161]:
def make_submission(preds: pd.DataFrame,
                    h: int) -> pd.DataFrame:
    wide = preds.pivot_table(index='id',
                             columns='date',
                             observed=True)
    # wide.columns = [f'F{i+1}' for i in range(h)]
    wide.columns = [f'd_{TEST_START + i}' for i in range(h)]
    wide.columns.name = None
    wide.index.name = 'id'
    return wide

submission = make_submission(preds, HORIZON)
submission = submission.join(df[id_cols].set_index("id").drop_duplicates())
# submission.to_csv("data/ml_benchmark.csv")
submission.reset_index().rename(columns={"id": "unique_id"}).to_parquet("data/ml_baseline.snap.parquet")

In [162]:
submission

,d_1942,d_1943,d_1944,d_1945,d_1946,d_1947,d_1948,d_1949,d_1950,d_1951,...,d_1965,d_1966,d_1967,d_1968,d_1969,item_id,dept_id,cat_id,store_id,state_id
id,,,,,,,,,,,,,,,,,,,,,
FOODS_1_001_CA_1_evaluation,0.701140,0.858599,0.774327,0.817852,0.957635,1.233709,1.188126,0.838755,0.786562,0.774742,...,0.754253,0.748211,0.877114,1.103670,1.099405,FOODS_1_001,FOODS_1,FOODS,CA_1,CA
FOODS_1_001_CA_2_evaluation,0.729126,0.799108,0.785350,0.818576,0.960839,1.197364,1.261475,0.782686,0.798074,0.869764,...,0.866279,0.898848,0.984810,1.247570,1.297708,FOODS_1_001,FOODS_1,FOODS,CA_2,CA
FOODS_1_001_CA_3_evaluation,0.977203,0.877520,0.859335,0.831788,1.051413,1.175588,1.223946,0.906082,0.933188,1.033952,...,0.935167,0.929125,1.061426,1.269831,1.257585,FOODS_1_001,FOODS_1,FOODS,CA_3,CA
FOODS_1_001_CA_4_evaluation,0.178846,0.228902,0.295045,0.320118,0.355710,0.395775,0.421469,0.330799,0.317200,0.364547,...,0.361051,0.380703,0.373739,0.402344,0.404603,FOODS_1_001,FOODS_1,FOODS,CA_4,CA
FOODS_1_001_TX_1_evaluation,0.593642,0.905085,0.765304,0.800519,0.898751,1.040235,1.112625,0.813078,0.789266,0.742996,...,0.711239,0.697510,0.790894,0.983039,0.979863,FOODS_1_001,FOODS_1,FOODS,TX_1,TX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
HOUSEHOLD_2_516_TX_2_evaluation,0.215575,0.242546,0.275643,0.275643,0.298782,0.358378,0.378982,0.281818,0.278832,0.288368,...,0.292560,0.292585,0.336693,0.395477,0.393225,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,TX_2,TX
HOUSEHOLD_2_516_TX_3_evaluation,0.353509,0.314403,0.318199,0.290450,0.315385,0.353773,0.394723,0.339277,0.353372,0.364913,...,0.320801,0.325689,0.352665,0.398563,0.409801,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,TX_3,TX
HOUSEHOLD_2_516_WI_1_evaluation,0.159356,0.181270,0.230615,0.263766,0.329102,0.368062,0.368062,0.266955,0.266955,0.276094,...,0.280683,0.280708,0.350078,0.411930,0.411933,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,WI_1,WI


In [163]:
print(f"""Notebook finished in {(time.time() - start_time) / 60:.2f}m""")

Notebook finished in 214.08m
